# Demo App: End-to-End


This is the final notebook. It is the one a user would actually run. The Flet app under `src/notebooks/recsys/ui/app.py` ties the prior eight stages — data, features, retrieval, ranking, feature-store online/offline parity, off-policy eval, feedback logging, metrics — into a single window: the user onboards (rates a few movies), picks a model from the dropdown, watches the top-10 list update live, rates or skips items they recognize or want to investigate, and follows the live metric snapshot update on every call.

Most of this notebook shows what is happening *inside* `recsys.ui.app`. The last cell actually launches the Flet desktop app, which you can close after a few minutes exploring the recommendations and toggling models.


## Setup


In [ ]:
#| echo: false
import warnings
warnings.filterwarnings("ignore")
import os; os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")


In [ ]:
import time
import numpy as np
import pandas as pd
import torch; torch.manual_seed(0)

import notebooks.recsys.ui.app as rec_app
from notebooks.recsys.config import MovieLensConfig
from notebooks.recsys.metrics import Metricator
from notebooks.recsys.models.classic import ALSRecommender, ContentRecommender
from notebooks.recsys.models.retrieval import (
    TwoTower, TwoTowerConfig, InBatchSoftmaxLoss, TwoTowerTrainer, Retriever,
)
from notebooks.recsys.models.ranking import build_ranker, RankerConfig, RankerTrainer, Ranker
from notebooks.recsys.models.llm_rerank import LLMRerankConfig, LLMReranker
from notebooks.recsys.features import FeatureStore


## How `RecAppBoot` builds the demo


When the app starts, `RecAppBoot.setup()` does a long-form `__init__`:

1. Load MovieLens (`load_movielens` from REC:01),
2. Split train/val with `time_split` (REC:01),
3. Build `FeatureStore` (REC:02),
4. Train ALS + a content ranker + a Two-Tower retriever + a DCN pairwise ranker (REC:03–REC:05),
5. Stash trained models in a `dict` keyed by name. (The MLflow backed registry in REC:07 persists these to disk; the demo app keeps everything in-memory so it starts in seconds.)

Calling `rec_app._boot()` returns a single shared instance — second-call cheap. **Cold-start path with no user ratings falls back to popularity**. With ≥5 ratings the app finds the *closest proxy user* via cosine sim over genre histograms and uses their rated data as the user's session evidence — a hack realistic production would replace with proper on-line re-embedding.


In [ ]:
t0 = time.time()
boot = rec_app._boot()
print(f'boot time: {time.time()-t0:.2f}s')
print('available models:', list(boot.models.keys()))


## Inspecting the four models' Recall@10 in one line

When the app boots it shows each model's metrics in a panel. Let us reproduce the numbers here for the notebook.


In [ ]:
metricator = Metricator(boot.val)
rows = []
for name, model in boot.models.items():
    try:
        fn = lambda u, k=10, m=model: m.recommend(u, k=k)
    except TypeError:
        fn = lambda u, k=10, m=model: m.recommend(u, k)
    sc = metricator.evaluate(fn, k=10)
    rows.append({"model": name,
                 "recall@10": round(sc['recall@10'], 4),
                 "ndcg@10": round(sc['ndcg@10'], 4),
                 "coverage@10": round(sc['coverage@10'], 4),
                 "novelty@10": round(sc['novelty@10'], 4)})
pd.DataFrame(rows)


## The UI hierarchy


The Flet page renders four panels stacked vertically:

1. **Onboarding** — 12 movie cards (top by rating count from `train`). Each has 1★ / 3★ / 5★ buttons. The user must rate at least five before recommendations engage (otherwise the model defaults to popularity — fine, just not useful).
2. **Model picker & Refresh** — dropdown over `{ALS, content, twotower, ranker}`; pressing *Refresh* calls `session.recommend(model_name, k=10)`.
3. **Top-10 recommendations** — each card has 1★ / 3★ / 5★ buttons plus a *skip* link. Rating or skipping writes to `session.feedback_log` (REC:07-style).
4. **Live metric panel** — recomputes the model's Recall@10 / NDCG@10 / Coverage@10 / Novelty@10 over `val`, with a footer reporting the session's spent events. These numbers are not affected by the session's new ratings — they use the offline training and validation — but the metric *panel in production* would shift incrementally with each new feedback row.


## Launching the Flet app


Two ways to start the demo. From a terminal:

```bash
uv run flet run src/notebooks/recsys/ui/app.py
```

Or from this notebook. Flet returns when the window is closed, so running this cell will hold the kernel until you close the window:


In [ ]:
import flet as ft
# We re-use the boot we already did in the cell above; the boot's caching
# prevents a second load_data / train loop.
# Uncomment the next line when actually running this notebook:
# ft.app(target=rec_app.main)
print("App launcher wired up")


::: {.callout-warning}
If the cell looks hung it is because Flet keeps the kernel academy until the window closes; nothing is wrong. Closing the window returns control to the next cell.
:::


## What is missing (linking back)


Recsys app = the closest you can get to *demo in one file*. But:

- **The session state is in-process.** Refreshing the page loses its ratings. Real production uses identity primitives (REC:07's `/v1/feedback` would take over).
- **Closest-proxy-user hack.** Real production re-embeds the new user into the two-tower model online (one forward pass through the user tower) and uses that for retrieval. We don't because it requires the user tower to accept a feature dict instead of an ID; this is a left exercise.
- **No persistence.** Restart loses all state. REC:07 has a disk-backed registry; REC:08 logs events to sqlite for off-policy analysis. None of these are wired to this demo by default; toggling them in is one-line constructor swaps.
- **Refresh metadata.** The model vectors never re-index the FAISS store. Restart the app gets you a fresh FAISS over the same trained tower. Adding new items in real time requires a streaming FAISS replacement or a periodic re-index job (REC:04's caveats).

From this point you have a working recommendation-system course. Each REC:0X notebook downloads its own data, trains models, computes metrics you can read off-line, and ends with a *deployable* artifact — the trained model — saved to the registry, ready to be served over FastAPI. Course complete.
